# U2T02: train SimCSE

BERT encodes tokens well and sentences badly. Mean-pooling `bert-base-uncased` and comparing sentences by cosine similarity makes almost everything look similar to everything else. SimCSE (Gao, Yao and Chen, 2021) fixes that with a contrastive loss. In the unsupervised version the only data augmentation is dropout. In the supervised version the positive is an SNLI entailment and the hard negative is a contradiction.

This notebook is the assignment, in order. It uses the Colab runtime as it is.

Colab cannot upload a folder. Upload the file `dropout-is-all-you-need.zip` (code and the SNLI file only, about 20 MB). Do not upload `.venv` or `runs/`: the virtualenv is for the Mac, and Colab will train its own checkpoints. The first code cell unzips that file.

You will not match the paper's STS-B numbers. The data here is a 100k SNLI subset, not the paper's 1M Wikipedia sentences or its SNLI+MNLI triplets. Explaining that gap is part of the work.

Put your team member names in the report before you submit.


## 0. Unzip the project

In the Colab file browser, upload `dropout-is-all-you-need.zip` into `/content`. The next cell unpacks it. You do not upload the folder itself.

Colab's own runtime is used. The cell installs `sentence-transformers` and `datasets` only if they are missing, then writes the versions it actually imported to `requirements-colab.txt`.

Set `USE_DRIVE = True` in that cell if you want checkpoints on Google Drive. A disconnect then keeps `runs/`. Leave it `False` when you are not on Colab.


In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd() / "dropout-is-all-you-need", Path("/content/dropout-is-all-you-need")]
root = next((path for path in candidates if (path / "simcse" / "model.py").exists()), None)
if root is None:
    zip_path = next(
        (path for path in (Path("/content/dropout-is-all-you-need.zip"), Path.cwd() / "dropout-is-all-you-need.zip") if path.exists()),
        None,
    )
    if zip_path is None:
        raise FileNotFoundError(
            "Upload dropout-is-all-you-need.zip to /content. Colab cannot upload a folder."
        )
    shutil.unpack_archive(zip_path, "/content")
    root = Path("/content/dropout-is-all-you-need")
    if not (root / "simcse" / "model.py").exists():
        raise FileNotFoundError(f"Unzipped {zip_path}, but {root} has no simcse/model.py.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

data_file = root / "data" / "snli_train_100k.jsonl"
if not data_file.exists():
    raise FileNotFoundError(f"Missing {data_file}. Every team trains on this file.")
print("Project:", root.resolve())
print("SNLI file:", data_file, f"({data_file.stat().st_size / 1e6:.1f} MB)")

# Set True on Colab if you want runs/ to survive a disconnect.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_runs = Path("/content/drive/MyDrive/simcse-runs")
    drive_runs.mkdir(parents=True, exist_ok=True)
    local_runs = Path("runs")
    if not local_runs.is_symlink():
        if local_runs.exists():
            for item in local_runs.iterdir():
                dest = drive_runs / item.name
                if dest.exists():
                    continue
                if item.is_dir():
                    shutil.copytree(item, dest)
                else:
                    shutil.copy2(item, dest)
            shutil.rmtree(local_runs)
        local_runs.symlink_to(drive_runs, target_is_directory=True)
print("Checkpoints:", Path("runs").resolve())

needed = []
for module, package in (
    ("sentence_transformers", "sentence-transformers==3.1.1"),
    ("datasets", "datasets==2.21.0"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        needed.append(package)
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needed])

tracked = {
    "torch", "transformers", "sentence-transformers", "datasets", "scipy", "numpy",
    "pandas", "matplotlib", "huggingface-hub", "scikit-learn", "safetensors", "tqdm",
}
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
pinned = []
for line in freeze.splitlines():
    name = line.split("==", 1)[0].lower().replace("_", "-")
    if name in tracked:
        pinned.append(line)
pinned.sort()
Path("requirements-colab.txt").write_text("\n".join(pinned) + "\n")
print("Wrote requirements-colab.txt")
for line in pinned:
    print(" ", line)

def run_name(mode, seed, same_dropout=False, no_hard_negatives=False, data_fraction=1.0):
    """Must match the default name built in train.py."""
    if mode == "unsupervised":
        batch_size, lr, hard = 64, 3e-5, False
    else:
        batch_size, lr, hard = 128, 5e-5, not no_hard_negatives
    return (
        f"{mode}_s{seed}_bs{batch_size}_lr{lr:g}_tau{0.05:g}"
        f"_frac{data_fraction:g}_samedrop{int(same_dropout)}_hardneg{int(hard)}"
    )

def train_if_needed(mode, seed, same_dropout=False, no_hard_negatives=False, data_fraction=1.0):
    """Skip a run that already has a saved best checkpoint."""
    name = run_name(mode, seed, same_dropout, no_hard_negatives, data_fraction)
    best = Path("runs") / name / "best"
    if best.is_dir() and any(best.iterdir()):
        print(f"skip {name}: checkpoint already exists")
        return
    command = [
        sys.executable, "train.py",
        "--mode", mode,
        "--seed", str(seed),
        "--data_fraction", str(data_fraction),
    ]
    if same_dropout:
        command.append("--same_dropout")
    if no_hard_negatives:
        command.append("--no_hard_negatives")
    subprocess.check_call(command)


## Part 1. The objective

Equation 1, unsupervised. A batch of N sentences is encoded twice. Dropout makes the two passes different, so `(h_i, h_i+)` is the positive pair. The other sentences in the batch are the negatives.

```
L_i = -log  exp(sim(h_i, h_i+) / tau)  /  sum_{j=1}^{N} exp(sim(h_i, h_j+) / tau)
```

- The numerator is the positive pair: the anchor and the same sentence under a second dropout mask.
- The denominator sums over every row in the batch, including the positive.
- One step therefore pushes the positive away from **N - 1** negatives.

We use **N = 64** for unsupervised training, the paper's BERT-base setting, so that number is **63**. A random batch has nearly equal similarities, so the loss starts near `ln(64) = 4.16`. If it falls to about 0 on the first step, the two views are identical: dropout is off, or the model is in `eval()`.

Equation 5, supervised. The positive is the entailment hypothesis. The candidate list is the N positives concatenated with the N contradiction hard negatives, so the denominator has 2N terms.

```
L_i = -log exp(sim(h_i, h_i+) / tau) / sum_j [ exp(sim(h_i, h_j+) / tau) + exp(sim(h_i, h_j-) / tau) ]
```

With **N = 128** (the paper uses 512; 128 is what fits on a T4 at the paper's learning rate of 5e-5), a complete batch has **2N - 1 = 255** negatives: the other 127 positives, plus all 128 hard negatives. This file only has a contradiction for 28.4% of entailment pairs, and those missing columns are masked out, so a typical step pushes against about 127 + 36 negatives. The denominator still doubles in structure, because every row contributes a positive candidate and a hard-negative candidate.

Temperature `tau` divides the cosine before the exponential. Small `tau` makes the softmax peaky and the gradient concentrates on the hardest negatives. The paper's supervised ablation (Table D.1, STS-B dev) peaks at `tau = 0.05` (86.2). `tau = 1` flattens the softmax and the score falls to 64.0. Cosine with `tau = 0.05` also beats an unscaled dot product (85.9).

Sentence-BERT (2019) trains a three-way classifier on NLI pairs, then throws the classifier away and scores STS-B with cosine. SimCSE trains the cosine directly, with in-batch negatives. That matters because STS-B here has no regressor on top: the ranking of the cosines has to match the human ranking. The contrastive denominator is also what improves uniformity. The numerator pulls positives together (alignment). The sum in the denominator penalizes high similarity with negatives, which flattens the dominant singular direction of an anisotropic BERT space (uniformity). Unsupervised SimCSE mostly fixes uniformity, because the two dropout views are already well aligned. Supervised entailment pairs then tighten alignment further.


In [ ]:
import math
import torch
import torch.nn.functional as F
from simcse.model import info_nce, info_nce_with_hard_negatives

# Orthogonal sentences: the positive cosine is 1 and every negative cosine is 0.
# Eq. 1 then has a closed form, which is the check that the loss is the one in the paper.
N = 64
eyes = torch.eye(N)
loss = info_nce(eyes, eyes, tau=0.05)
expected = -math.log(math.exp(1 / 0.05) / (math.exp(1 / 0.05) + (N - 1) * math.exp(0)))
print(f"Eq. 1 loss: {loss.item():.6f}")
print(f"closed form: {expected:.6f}")
print(f"negatives pushing on the positive: {N - 1}")
print(f"loss of a uniform softmax, ln(N): {math.log(N):.3f}")

# Same cosine pattern, four temperatures. Smaller tau is peakier, so the loss drops.
similarities = torch.full((8, 8), 0.7)
similarities.fill_diagonal_(0.9)
labels = torch.arange(8)
print("\ntemperature    loss")
for tau in (0.01, 0.05, 0.1, 1.0):
    print(f"{tau:<14} {F.cross_entropy(similarities / tau, labels).item():.3f}")

# Eq. 5 with every hard negative present: 2N candidates, 2N - 1 of them negatives.
# Masking every hard negative returns Eq. 1. That is what we do for the 72% of
# pairs that have no contradiction.
batch = 4
vectors = F.normalize(torch.randn(batch, 8), dim=-1)
present = torch.ones(batch, dtype=torch.bool)
absent = torch.zeros(batch, dtype=torch.bool)
with_negatives = info_nce_with_hard_negatives(vectors, vectors, torch.randn_like(vectors), present, 0.05)
masked = info_nce_with_hard_negatives(vectors, vectors, vectors, absent, 0.05)
plain = info_nce(vectors, vectors, 0.05)
print(f"\nEq. 5 negatives when every row has a hard negative: {2 * batch - 1}")
print(f"masked hard negatives match Eq. 1: {torch.allclose(masked, plain, atol=1e-4)}")
print(f"Eq. 5 loss (finite): {with_negatives.item():.4f}")


Checkpoint. Say, in one sentence, why the supervised denominator has twice as many terms as the unsupervised one. Then say what changes about that count in our file, where only 28.4% of premises have a contradiction.


## Part 2. The data

One file, `data/snli_train_100k.jsonl`: 100,000 records of premise, hypothesis, and label. Label 0 is entailment, 1 is neutral, 2 is contradiction.

From that file:

- Unsupervised: the unique sentences across premises and hypotheses. The assignment count is **165,529**. Do not strip whitespace. Stripping drops one sentence and the count stops matching.
- Supervised: every label-0 record, **33,351** pairs, not the 33,333 unique pairs. Eighteen duplicate records stay.
- Hard negative: the first contradiction hypothesis for that premise, in file order. **9,488** pairs have one (28.4%). Pairs without one stay in the training set. Their hard-negative column is masked inside the loss, so it does not enter the denominator and gets no gradient. Dropping those pairs would change the training set the ablation is compared against.

STS-B comes from `sentence-transformers/stsb`. Dev (`validation`) has 1,500 pairs and is the only split used to pick a checkpoint. Test has 1,379 pairs and is scored once, at the end. The Hub stores the gold score in [0, 1]; the loader multiplies by 5 so the rest of the notebook can talk about the original 0-5 ratings. Spearman does not change.


In [ ]:
from simcse.data import dataset_stats, load_snli_rows, supervised_pairs

stats = dataset_stats()
for key, value in stats.items():
    print(f"{key}: {value}")
if not stats["counts_match_assignment"]:
    raise RuntimeError("Counts do not match the assignment. Fix the dataset construction before training.")

pairs = supervised_pairs(load_snli_rows())
print("\nAn entailment pair that has a contradiction:")
for pair in pairs:
    if pair["hard_negative"] is not None:
        print("  premise:      ", pair["premise"])
        print("  entailment:   ", pair["positive"])
        print("  contradiction:", pair["hard_negative"])
        break
print("\nAn entailment pair with no contradiction (this column is masked in Eq. 5):")
for pair in pairs:
    if pair["hard_negative"] is None:
        print("  premise:   ", pair["premise"])
        print("  entailment:", pair["positive"])
        break


## Part 4, before any training. Check the evaluator

The assignment is explicit: score a fixed model before the first training run. If your number is far from the notes, the bug is in the evaluator, not in the trainer.

Protocol, for every model, including these two: embed both sentences, L2-normalize, take the cosine, correlate with the human scores using Spearman, multiply by 100. No regressor. Appendix B of the paper is why. Mean pooling has to ignore padding tokens. The Hub STS-B score is rescaled to 0-5 inside the loader; ranks are unchanged.

| model | dev | test |
| --- | --- | --- |
| `bert-base-uncased`, mean pool | 59.31 | 47.29 |
| `sentence-transformers/bert-base-nli-mean-tokens` | 80.77 | 76.98 |

Alignment is the mean squared distance between the two sentences of a pair whose gold score is at least 4. Uniformity is `log mean exp(-2 ||f(x) - f(y)||^2)` over the distinct sentences in the split, with embeddings on the unit sphere. Lower is better for both.


In [ ]:
from simcse.evaluate import check_references

reference = check_references()
reference["deltas"]


Checkpoint. Both deltas should be well under 0.2. If they are not, stop. Look at padding masks in mean pooling before you spend hours training.


## Part 3. Train both modes

Both modes start from `bert-base-uncased`. Training always pools the [CLS] token through BERT's pooler (a linear layer and a tanh). That is the MLP. Unsupervised SimCSE drops the MLP at evaluation and uses the raw [CLS] state. Supervised SimCSE keeps the MLP at evaluation. The paper finds that this split works better.

Shared choices, listed for the report:

| | unsupervised | supervised |
| --- | --- | --- |
| batch size | 64 | 128 (paper: 512) |
| learning rate | 3e-5 | 5e-5 |
| epochs | 1 | 3 |
| temperature | 0.05 | 0.05 |
| max length | 32 | 32 |
| dropout | 0.1 | 0.1 |
| optimizer | AdamW, weight decay 0, no warmup, linear decay | same |
| dev eval | every 250 steps, keep the best | same |

The learning rate for supervised training is the one the paper tuned at batch 512. It is not retuned for 128. That is a setup choice, and it belongs in the gap paragraph.

`train.py` never loads the test split. Each run writes `runs/<run_name>/config.json` and appends a row to `runs/log.csv`.

Run the short cell first. The first loss should be near 4.16. Dev Spearman will still be weak after 20 steps; the point of this cell is the loss, not the score.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "train.py",
    "--mode", "unsupervised",
    "--seed", "42",
    "--max_steps", "20",
    "--eval_steps", "20",
    "--output_dir", "runs/notebook_smoke",
    "--run_name", "smoke_unsup",
])


If that loss looked right, train the real models. A run is skipped when `runs/<name>/best` already exists, so re-running this notebook after a disconnect does not start that run over. Three seeds, so the later ablation can be compared with run-to-run noise.


In [ ]:
for seed in (42, 43, 44):
    train_if_needed("unsupervised", seed)


In [ ]:
for seed in (42, 43, 44):
    train_if_needed("supervised", seed)


Checkpoint. Why does a smaller batch hurt the supervised run more than the unsupervised one? The supervised denominator was built to hold 2N candidates. Cutting N from 512 to 128 removes both in-batch entailments and in-batch contradictions, and the learning rate was chosen for the larger batch.


## Part 5. One ablation per mode

Change exactly one thing, and keep seed 42 so the data order matches the main run.

- Unsupervised: `--same_dropout`. The sentence is encoded once and the vector is reused, so both views share one mask. Predict the direction before you read the number. In the paper's Table 3 this "fixed 0.1" setting scores 43.6 on STS-B dev, against 82.5 with two independent masks. The positive pair stops teaching alignment, because the two views are the same vector. The model can only spread points apart, and the space collapses.
- Supervised: `--no_hard_negatives`. The contradiction columns disappear. Batch size, learning rate, steps, and seed stay put. The paper's Table 4 moves from 84.9 (entailment only) to 86.2 once contradictions are added.

`summarize.py` compares each delta with the standard deviation of the three main seeds. A difference larger than that spread is not run-to-run noise.


In [ ]:
train_if_needed("unsupervised", 42, same_dropout=True)
train_if_needed("supervised", 42, no_hard_negatives=True)


## Part 6, first half. How much of the gap is "less of this file"?

The paper's unsupervised model trains on 1,000,000 Wikipedia sentences. Ours trains on 165,529 SNLI captions. The paper's supervised model trains on the SNLI+MNLI entailment pairs (about 314k in their Table 4; the assignment compares against about 275k triplets), and every row has a hard negative.

These two cells keep seed 42 and change only the fraction of our file. The 25% slice is a prefix of the 50% slice, which is a prefix of the full set. They do not measure the Wikipedia-versus-captions domain shift. They measure how much the score moves when this file gets smaller. The domain shift and the batch-size choice are whatever is left.


In [ ]:
for fraction in (0.25, 0.5):
    train_if_needed("unsupervised", 42, data_fraction=fraction)
    train_if_needed("supervised", 42, data_fraction=fraction)


## Score the test split once

Pick models by dev Spearman. Then score test. The script refuses to overwrite a test number that is already stored.

Score:

- the best unsupervised seed and the best supervised seed
- the same-mask ablation and the hard-negatives-off ablation, so the test delta exists

Leave the 25% and 50% runs on dev. They are there to explain the gap, not to pick a model.


In [ ]:
import subprocess
import sys
from pathlib import Path

from summarize import best_main, find_one, load_rows

rows = load_rows(Path("runs/log.csv"))
chosen = {
    "unsupervised": best_main(rows, "unsupervised")["run_name"],
    "supervised": best_main(rows, "supervised")["run_name"],
    "same_dropout": find_one(
        rows,
        lambda row: row["mode"] == "unsupervised" and row["seed"] == "42"
        and row["same_dropout"] in {"True", "true", "1"} and float(row["data_fraction"]) == 1.0,
    )["run_name"],
    "no_hard_negatives": find_one(
        rows,
        lambda row: row["mode"] == "supervised" and row["seed"] == "42"
        and row["hard_negatives"] in {"False", "false", "0"} and float(row["data_fraction"]) == 1.0,
    )["run_name"],
}
print(chosen)
for name in chosen.values():
    subprocess.check_call([sys.executable, "score_test.py", "--run-dir", f"runs/{name}"])


## Part 4, after training. Alignment, uniformity, and a failure case

For each model the notebook plots alignment against uniformity, and the cosine of each STS-B test pair grouped by the human rating (0-1, 1-2, 2-3, 3-4, 4-5). A model that ranks meaning should show the boxes stepping up as the human score steps up.

The retrieval cell also writes two pairs you have to discuss:

- false friend: gold score below 1, and the highest cosine in that bin
- missed paraphrase: gold score at least 4, and the lowest cosine in that bin

Read the sentences. Lexical overlap, negation, and numbers are the usual reasons a cosine lies.


In [ ]:
import subprocess
import sys
from pathlib import Path
import json

from summarize import best_main, load_rows

rows = load_rows(Path("runs/log.csv"))
unsup = best_main(rows, "unsupervised")["run_name"]
sup = best_main(rows, "supervised")["run_name"]
subprocess.check_call([
    sys.executable, "analysis.py",
    "--include-references",
    "--split", "test",
    "--checkpoint", f"unsup=runs/{unsup}/best",
    "--checkpoint", f"sup=runs/{sup}/best",
])
retrievals = json.loads(Path("runs/analysis/retrievals.json").read_text())
for name, payload in retrievals["models"].items():
    print("\n", name)
    print("  false friend:", payload["failures"]["false_friend"])
    print("  missed paraphrase:", payload["failures"]["missed_paraphrase"])


The figures are `runs/analysis/alignment_uniformity.png` and `runs/analysis/similarity_by_rating.png`.


In [ ]:
from IPython.display import Image, display
display(Image("runs/analysis/alignment_uniformity.png"))
display(Image("runs/analysis/similarity_by_rating.png"))


## Part 5 and Part 6. The tables

The ablation summary says whether each delta is larger than the seed standard deviation. The benchmark table is the one the report needs. Paper numbers, checked against Gao, Yao and Chen 2021:

- unsupervised, STS-B dev 82.5 (Table 3), test 76.85 (Table 5)
- supervised, entailment plus hard negatives, STS-B dev 86.2 (Table 4), test 84.25 (Table 5)

Compare dev with dev and test with test. Alignment and uniformity for the paper's models are not in the paper as STS-B scalars, so those cells stay blank. Ours are computed on the test split.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

from summarize import as_float, is_main, load_rows

subprocess.check_call([sys.executable, "summarize.py"])
summary = json.loads(Path("runs/ablation_summary.json").read_text())
print("\nAblations")
print(pd.DataFrame(summary["ablations"]).to_string(index=False))
print("\nData fractions (dev only)")
print(pd.DataFrame(summary["data_fractions"]).to_string(index=False))

reference = json.loads(Path("runs/reference_eval.json").read_text())
rows = load_rows(Path("runs/log.csv"))

def best(mode):
    group = [row for row in rows if row["mode"] == mode and is_main(row)]
    return max(group, key=lambda row: as_float(row["best_dev_spearman"]) or -1)

table = []
for label, row in (("SimCSE unsupervised (ours)", best("unsupervised")), ("SimCSE supervised (ours)", best("supervised"))):
    table.append({
        "model": label,
        "dev": round(as_float(row["best_dev_spearman"]), 2),
        "test": None if as_float(row["test_spearman"]) is None else round(as_float(row["test_spearman"]), 2),
        "alignment": None if as_float(row["alignment"]) is None else round(as_float(row["alignment"]), 4),
        "uniformity": None if as_float(row["uniformity"]) is None else round(as_float(row["uniformity"]), 4),
    })

for label, key in (("bert-base-uncased mean", "bert-base-uncased-mean"), ("SBERT-2019", "sbert-2019")):
    dev = reference[key]["splits"]["validation"]
    test = reference[key]["splits"]["test"]
    table.append({
        "model": label,
        "dev": round(dev["spearman"], 2),
        "test": round(test["spearman"], 2),
        "alignment": round(test["alignment"], 4),
        "uniformity": round(test["uniformity"], 4),
    })

table.append({"model": "SimCSE unsupervised (paper)", "dev": 82.5, "test": 76.85, "alignment": None, "uniformity": None})
table.append({"model": "SimCSE supervised (paper)", "dev": 86.2, "test": 84.25, "alignment": None, "uniformity": None})
benchmark = pd.DataFrame(table)
benchmark.to_csv("runs/benchmark_table.csv", index=False)
benchmark


### The gap paragraph

Write this in the report, with the numbers from the table above. Name the differences, then say how much of the distance they account for.

Data, not your code:

- Unsupervised training here is 165,529 SNLI captions. The paper uses 1,000,000 Wikipedia sentences: more data, and a different domain and length. STS-B is not Wikipedia.
- Supervised training here is 33,351 SNLI entailment pairs, and a contradiction exists for only 9,488 of them. The paper uses SNLI+MNLI, on the order of 275k triplets, with a hard negative on every row. Their Table 4 dev score for that full set is 86.2, against 84.9 with no hard negatives.
- One unsupervised epoch here is about 2,586 steps. One epoch on 1M sentences at batch 64 is about 15,600 steps.

Setup, which is yours:

- Supervised batch size 128 instead of 512, with the learning rate left at 5e-5.
- Whatever else you changed and recorded in `runs/<run>/config.json`.

Use the 25% and 50% dev scores to put a number on quantity inside this file. If going from 50% to 100% moves the dev score by a point, and you are still several points short of 82.5 or 86.2, most of the gap is not "we used half the file". The rest is the domain (Wikipedia versus SNLI captions), the missing MNLI pairs, the missing hard negatives, and the smaller supervised batch. The hard-negative ablation is the number for that last data difference: report its dev and test delta next to the seed standard deviation.


## Part 7. Publish

Export the best model of each mode as a sentence-transformers model.

- Unsupervised: Transformer plus CLS pooling. The MLP stays in the BERT weights and is not applied. That matches "drop the MLP at test time".
- Supervised: the same, plus a `Dense` layer whose weight and bias are the BERT pooler, with a tanh. That is the MLP kept at test time.

The export reloads the saved folder and checks that the test Spearman matches the evaluator. Nothing is uploaded until you run the push cell, and that cell refuses to push until `HUB_REPOS` names your Hugging Face user. The model card records the training file, the recipe, the metrics, and the limitations: English, uncased, SNLI captions rather than Wikipedia, 28.4% hard-negative coverage, batch 128 rather than 512.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import subprocess
import sys
from pathlib import Path

from summarize import best_main, load_rows

# Only replace the two YOUR_HF_USER strings below. Leave the if line as it is.
HUB_REPOS = {
    "unsupervised": "YOUR_HF_USER/simcse-unsup-snli100k",
    "supervised": "YOUR_HF_USER/simcse-sup-snli100k",
}

rows = load_rows(Path("runs/log.csv"))
for mode in ("unsupervised", "supervised"):
    repo_id = HUB_REPOS[mode]
    if "YOUR_HF_USER" in repo_id:
        raise RuntimeError("Set HUB_REPOS to your Hugging Face username before pushing.")
    run_name = best_main(rows, mode)["run_name"]
    subprocess.check_call([
        sys.executable, "export_and_push.py",
        "--checkpoint", f"runs/{run_name}/best",
        "--mode", mode,
        "--out", f"exported_models/{mode}",
        "--repo-id", repo_id,
        "--push",
    ])


Checkpoint. The test Spearman after the Hub reload has to match the benchmark table. A mismatch is almost always the pooling or the MLP: unsupervised must not apply the pooler, supervised must.

## Part 8. Optional

More data, another dataset, or another ablation is extra. If you do it, say so in the report and keep the same rule: one change at a time, dev for decisions, test once.

## What the report has to contain

- Team member names.
- The Part 1 answers, including 63 unsupervised negatives and the supervised 2N - 1 count at batch 128.
- The configuration table from Part 3.
- The benchmark table, the two plots, the retrievals, and a discussion of at least one failure.
- Both ablations, with dev and test deltas, and whether each delta beats the seed standard deviation.
- The gap paragraph, with numbers from the fraction runs and the hard-negative ablation.
- Anything that broke: a loss that sat still, a loss that hit zero, a reference score that missed. What you changed belongs there too.
